# Funciones universales y estadística avanzada con NumPy

Notebook guiado que acompaña a `13-numpy-ufuncs-estadistica-avanzado.md`.

## Qué es una ufunc

Una *universal function* (ufunc) es una operación **vectorizada** que se aplica a cada
elemento de uno o más arrays al mismo tiempo, con soporte de *broadcasting* y salida
opcional. Está implementada en C, por lo que evita bucles de Python.

In [2]:
import numpy as np

x = np.array([1., 2., 4.])
print(np.sqrt(x))
print(np.exp(x))
print(np.log(x))

[1.         1.41421356 2.        ]
[ 2.71828183  7.3890561  54.59815003]
[0.         0.69314718 1.38629436]


`np.add`, `np.multiply`... son ufuncs, así como `np.sqrt`, `np.sin`, `np.exp`. Tienen
un método `.reduce`, `.accumulate` y `.outer` que los vuelven extremadamente útiles.

## Reducciones y acumulaciones: `reduce`, `accumulate`, `outer`

In [3]:
x = np.array([1, 2, 3, 4])
print('reduce suma:', np.add.reduce(x))
print('producto acumulado:', np.multiply.accumulate(x))
print('outer:')
print(np.outer(np.array([1, 2]), np.array([10, 20, 30])))

reduce suma: 10
producto acumulado: [ 1  2  6 24]
outer:
[[10 20 30]
 [20 40 60]]


- `reduce`: combina todos los elementos con la operación.
- `accumulate`: guarda los resultados parciales.
- `outer`: combina cada par de elementos (tabla).

Estas operaciones aparecen en cálculo de probabilidades conjuntas, sumas parciales y
matrices de interacciones.

## Redondeo y límites: `round`, `floor`, `ceil`, `clip`

Para controlar la precisión de resultados o restringir valores a un rango:

In [4]:
x = np.array([1.234, 2.567, -0.5])
print(np.round(x, 1))
print(np.floor(x))
print(np.ceil(x))

v = np.array([-2., 0.5, 3.9])
print(np.clip(v, 0, 1))

[ 1.2  2.6 -0.5]
[ 1.  2. -1.]
[ 2.  3. -0.]
[0.  0.5 1. ]


`clip` es muy usado para limitar probabilidades o predicciones a un rango válido.

## Estadística: percentiles, histograma, bincount

In [6]:
datos = np.random.default_rng(5).normal(loc=10, scale=3, size=500)
print('percentiles:', np.percentile(datos, [25, 50, 75]))
hist, bordes = np.histogram(datos, bins=10)
print('histograma:', hist)
print('bordes:', bordes.round(2))

percentiles: [ 8.00901008  9.90354083 12.10381342]
histograma: [  7  12  52  82 112  93  78  39  22   3]
bordes: [ 1.9   3.54  5.18  6.83  8.47 10.11 11.75 13.39 15.04 16.68 18.32]


- `percentile` da cuantiles; útil para describir distribución y outliers.
- `histogram` agrupa en intervalos; la base de muchas visualizaciones.
- `np.bincount` cuenta enteros no negativos (rápida para conteos):

In [7]:
enteros = np.array([0, 1, 1, 3, 2, 1, 3])
print('bincount:', np.bincount(enteros))

bincount: [1 3 1 2]


## `unique`, orden y selección de valores

In [8]:
valores = np.array([3, 1, 3, 2, 1, 2, 3])
v, c = np.unique(valores, return_counts=True)
print('valores:', v, 'frecuencias:', c)

valores: [1 2 3] frecuencias: [2 2 3]


Para clasificación y frecuencias de categorías, `unique(return_counts=True)` es
directo. Para estadística robusta, `median` y `percentile` son menos sensibles a
outliers que `mean`.

## Correlación y covarianza
`np.corrcoef` y `np.cov` cuantifican asociación lineal. Recuerda `rowvar=False` para
que cada **columna** sea una variable:

In [9]:
rng = np.random.default_rng(7)
horas = rng.uniform(0, 8, size=200)
resultado = 2.5 * horas + rng.normal(0, 2, size=200)
X = np.column_stack([horas, resultado])
print('correlación:\n', np.corrcoef(X, rowvar=False).round(3))
print('covarianza:\n', np.cov(X, rowvar=False).round(2))

correlación:
 [[1.    0.949]
 [0.949 1.   ]]
covarianza:
 [[ 5.48 13.7 ]
 [13.7  38.05]]


La correlación no implica causalidad; la covarianza depende de las unidades y la
correlación normaliza a `[-1, 1]`.

## Funciones lógicas y comparaciones como ufuncs
Las comparaciones (`np.greater`, `np.less`, `np.equal`) son ufuncs que devuelven arrays
booleanos. Se combinan con `&`, `|`, `~` y paréntesis:

In [11]:
x = np.array([1, 5, 9, 14])
print(np.greater(x, 7))
print((x > 3) & (x < 12))
print(np.logical_and(x > 3, x < 12))
x[np.greater(x, 7)]

[False False  True  True]
[False  True  True False]
[False  True  True False]


array([ 9, 14])

## Patrón: resumir y clasificar con cuartiles
Resumir la distribución con percentiles y clasificar sus valores por cuartiles con `select`. Este ejemplo no normaliza los datos a `[0, 1]`.

In [13]:
rng = np.random.default_rng(2)
datos = rng.normal(loc=50, scale=15, size=100)
minimo, q1, mediana, q3, maximo = np.percentile(datos, [0, 25, 50, 75, 100])
print('resumen:', minimo.round(2), q1.round(2), mediana.round(2), q3.round(2), maximo.round(2))



resumen: 13.38 39.6 49.36 60.02 85.62


Con `np.select` se clasifica en categorías:

In [14]:
etiquetas = np.select(
    [datos < q1, datos < mediana, datos < q3],
    ['bajo', 'medio', 'alto'],
    default='muy alto',
)
print(np.unique(etiquetas, return_counts=True))

(array(['alto', 'bajo', 'medio', 'muy alto'], dtype='<U8'), array([25, 25, 25, 25]))


## Errores frecuentes

1. Usar `np.round` esperando redondeo "a la mitad" idéntico a otras herramientas
  (NumPy usa redondeo de banquero en algunos casos).
2. Confundir `histogram` (devuelve conteos + bordes) con `bincount` (conteo de enteros).
3. Olvidar `rowvar=False` en `corrcoef`/`cov` cuando las variables son columnas.
4. Aplicar `mean` a datos con `nan` sin `nanmean` (o sin limpiar).
5. Usar `np.unique(..., return_counts=True)` olvidando desempaquetar dos valores.
6. No poner paréntesis en combinaciones lógicas con `&`/`|`.

## Ejercicios

1. Calcula `sqrt`, `exp` y `log` de un array y observa formas.
2. Usa `multiply.accumulate` para el producto parcial de `[1..5]`.
3. Acota probabilidades a `[0,1]` con `clip`.
4. Obtén percentiles 5, 25, 50, 75, 95 de 1000 valores.
5. Cuenta frecuencias con `unique(return_counts=True)`.
6. Clasifica un array en 4 tramos con `np.select` y los cuartiles.